# filter the cell_types

In [ ]:
import scanpy as sc

adata = sc.read_h5ad('lung_disease.h5ad')

In [ ]:
# adata.obs['disease'],adata.obs['assay']=='10x 3\' v3'
sc.pl.umap(adata, color=['cell_type'], wspace=0.4,save_path='umap_cell_type.png',dpi=600)
plt.savefig()

In [ ]:
# adata.obs['disease'].value_counts()

adata[(adata.obs['disease']=='COVID-19') | (adata.obs['disease']=='normal'), :]

In [ ]:
adata.obs['assay'].value_counts()
# adata[(adata.obs['assay']=="10x 5' transcription profiling") :]

In [ ]:
cov_norm_adata = adata[(adata.obs['assay']=="10x 3' v3") & ((adata.obs['disease']=="normal") | (adata.obs['disease']=="COVID-19"))]

In [ ]:
sc.pp.highly_variable_genes(cov_norm_adata,n_top_genes=2000,subset=True)

In [ ]:
cov_norm_adata.write_h5ad('lung_cov_norm.h5ad')

In [ ]:
import scanpy as sc
cov_norm_adata=sc.read_h5ad('lung_cov_norm.h5ad')

In [ ]:
import matplotlib.pyplot as plt
# sc.pl.umap(cov_norm_adata,color=['disease'],save_path='umap_disease.png',kdpi=600)

sc.pl.umap(cov_norm_adata, color=['disease'], show=False)  # show=False 只生成对象不弹窗
plt.savefig("umap.png", dpi=1000, bbox_inches="tight")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Sort by COVID-19 proportion (descending)
ct_disease_prop = pd.crosstab(
    cov_norm_adata.obs['cell_type'],
    cov_norm_adata.obs['disease'],
    normalize='index'
)
ct_disease_prop_sorted = ct_disease_prop.sort_values('COVID-19', ascending=False)

# Raw counts aligned with the sorted proportions
ct_counts = pd.crosstab(
    cov_norm_adata.obs['cell_type'],
    cov_norm_adata.obs['disease']
)
ct_counts_sorted = ct_counts.loc[ct_disease_prop_sorted.index]

# Horizontal stacked bar plot with spacing
sns.set_style("whitegrid")
colors = sns.color_palette("Set2", n_colors=ct_disease_prop_sorted.shape[1])
y_positions = np.arange(len(ct_disease_prop_sorted))
bar_height = 0.7
left = np.zeros(len(ct_disease_prop_sorted))

fig, ax = plt.subplots(figsize=(13, 20))

for (disease, color) in zip(ct_disease_prop_sorted.columns, colors):
    proportions = ct_disease_prop_sorted[disease].values
    counts = ct_counts_sorted[disease].values
    bars = ax.barh(
        y_positions,
        proportions,
        left=left,
        height=bar_height,
        color=color,
        edgecolor="white",
        linewidth=0.5,
        label=disease,
    )
    for bar, count in zip(bars, counts):
        if count == 0:
            continue
        x = bar.get_x() + bar.get_width() / 2
        y = bar.get_y() + bar.get_height() / 2
        r, g, b, _ = bar.get_facecolor()
        text_color = "black" if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else "white"
        ax.text(x, y, f"{int(count)}", ha="center", va="center", fontsize=8, color=text_color)
    left += proportions

ax.set_yticks(y_positions)
ax.set_yticklabels(ct_disease_prop_sorted.index)
ax.set_xlim(0, 1.12)
ax.set_xlabel("Proportion")
ax.set_ylabel("Cell Type")
ax.set_title("Proportion of Disease Types by Cell Type (Sorted by COVID-19)")
ax.legend(title="Disease", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="x", linestyle="--", alpha=0.4)

# Annotate total counts at the end of each bar
total_counts = ct_counts_sorted.sum(axis=1)
for y_pos, total in zip(y_positions, total_counts):
    ax.text(1.05, y_pos, f"Total: {int(total)}", ha="left", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

# Filter cell types with COVID-19 proportion between 40%-60% (after sorting)
filtered_cell_types = ct_disease_prop_sorted[
    (ct_disease_prop_sorted['COVID-19'] >= 0.4) & (ct_disease_prop_sorted['COVID-19'] <= 0.6)
]
print("Cell types with COVID-19 proportion between 40%-60%:")
print(filtered_cell_types)

# select multiciliated epithelial cell
#

In [ ]:
cov_norm_adata[cov_norm_adata.obs['cell_type']=='CD4-positive, alpha-beta T cell'].obs['disease'].value_counts()

In [ ]:
filtered_covid_adata = covid_adata[covid_adata.obs['cell_type'].isin(filtered_cell_types.index)]

In [ ]:
filtered_covid_adata

sc.pl.umap(filtered_covid_adata, color=['cell_type', 'disease'], wspace=0.4, ncols=2)

In [ ]:
# filtered_covid_adata.write_h5ad('lung_disease_filtered.h5ad')

filtered_covid_adata

# analyze the filtered_covid_adata

In [ ]:
import scanpy as sc
import numpy as np

sc.settings.verbosity = 3

file_name = 'lung_disease_filtered.h5ad'

old_adata = sc.read_h5ad(file_name)
old_adata.obs['cell_type'].value_counts()

In [ ]:
# cell_type
cell_type='pulmonary artery endothelial cell'


# create a new AnnData object
adata=sc.AnnData(X=old_adata[old_adata.obs['cell_type'] == cell_type].raw.X,raw=old_adata[old_adata.obs['cell_type'] == cell_type].raw)

# label cells and genes(var and var_names cannot have the same data)
adata.var_names=old_adata[old_adata.obs['cell_type'] == cell_type].var['feature_name']
adata.obs_names=old_adata[old_adata.obs['cell_type'] == cell_type].obs_names

adata.obs['disease']=old_adata[old_adata.obs['cell_type'] == cell_type].obs['disease'].values

# QC metrics
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata = adata[
    (adata.obs.n_genes_by_counts < 4500)
    & (adata.obs.n_genes_by_counts > 200)
    & (adata.obs.total_counts < 20000)
    & (adata.obs.pct_counts_mt < 20),
    :,
].copy()
adata.layers["counts"] = adata.X.copy()

In [ ]:
adata

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=10)

adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata,n_top_genes=2000,inplace=True)

In [ ]:
adata=adata[:,adata.var['highly_variable']]
sc.pp.scale(adata, max_value=10)

In [ ]:
import seaborn as sns

# Plot density plots for the top highly variable genes
import matplotlib.pyplot as plt

# Get the top 10 most highly variable genes
# top_genes = adata.var.sort_values('dispersions_norm', ascending=False).head(10).index
top_genes = adata.var.sort_values('means', ascending=False).head(40).index

# Create subplots for gene density visualization
fig, axes = plt.subplots(4, 10, figsize=(20, 8))
axes = axes.flatten()

for i, gene in enumerate(top_genes):
    # Get gene expression data
    gene_expression = adata[:, gene].X.toarray().flatten()
    
    # Calculate the percentage of cells expressing this gene
    expressing_cells = (gene_expression > 0).sum()
    total_cells = len(gene_expression)
    frequency = expressing_cells / total_cells * 100
    
    # Plot density plot
    sns.kdeplot(gene_expression, ax=axes[i], fill=True, alpha=0.7)
    axes[i].set_title(f'{gene}\n{frequency:.1f}% cells expressing')
    axes[i].set_xlabel('Expression Level')
    axes[i].set_ylabel('Density')

plt.tight_layout()
plt.show()


In [ ]:
cell_type='capillary endothelial cell'
seed=42
n_top_genes=2000
n_sample=8000

# create a new AnnData object
adata=sc.AnnData(X=old_adata[old_adata.obs['cell_type'] == cell_type].raw.X)
# label cells and genes(var and var_names cannot have the same data)
adata.var_names=old_adata[old_adata.obs['cell_type'] == cell_type].var['feature_name']
adata.obs_names=old_adata[old_adata.obs['cell_type'] == cell_type].obs_names

adata.obs['disease']=old_adata[old_adata.obs['cell_type'] == cell_type].obs['disease'].values
# adata.var_names_make_unique()
# adata.obs_names_make_unique()

adata

sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_genes(adata, min_cells=10)
# Quality control on count data
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

adata = adata[adata.obs.n_genes_by_counts < 7000, :]
adata = adata[adata.obs.pct_counts_mt < 15, :]


sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
np.random.seed(seed)
sc.pp.highly_variable_genes(adata,n_top_genes=n_top_genes)
adata=adata[:, adata.var.highly_variable]

# Select 1500 cells from each category in adata.obs['diabetes_model']
category_1 = adata.obs['disease'].unique()[0]
category_2 = adata.obs['disease'].unique()[1]
print(f"category_1: {category_1}, category_2: {category_2}")
print(f"category_1 cells: {sum(adata.obs['disease'] == category_1)}, category_2cells: {sum(adata.obs['disease'] == category_2)}")

if n_sample is None:
    adata.layers['lognorm'] = adata.X.toarray().copy()
else:
    cells_category_1 = np.random.choice(adata[adata.obs['disease'] == category_1].obs_names, n_sample, replace=False)
    cells_category_2 = np.random.choice(adata[adata.obs['disease'] == category_2].obs_names, n_sample, replace=False)
    selected_cells = np.concatenate([cells_category_1, cells_category_2])
    adata = adata[selected_cells]
    

#     adata.layers['lognorm'] = adata.X.toarray().copy()

#     return adata

In [ ]:
adata

In [ ]:
import numpy as np

# Count the number of False and True values
mt_counts = np.unique(adata.var_names.str.startswith("MT-"), return_counts=True)
mt_counts = dict(zip(mt_counts[0], mt_counts[1]))
print(mt_counts)



# Data Description

In [ ]:
import scanpy as sc
adata=sc.read_h5ad("lung_cov_norm.h5ad")
adata

In [ ]:
adata.obs['cell_type'].value_counts()
adata[adata.obs['sex']=='male'].write_h5ad('lung_cov_norm_male.h5ad')

In [ ]:
macro_subset = adata[adata.obs['cell_type'] == 'elicited macrophage']
macro_subset.obs

In [ ]:
macro_subset.obs['sex'].value_counts(),macro_subset.obs['development_stage'].value_counts()

In [ ]:
macro_subset[macro_subset.obs['sex']=='male'].obs['disease'].value_counts()

In [ ]:
sc.pl.umap(macro_subset, color=['disease'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42
# Count the occurrences of each diabetes status
status_counts = macro_subset.obs['disease'].value_counts()

# Create a bar plot
plt.figure(figsize=(6, 4))
sns.barplot(x=status_counts.index, y=status_counts.values, palette='Set2')
plt.xlabel('Disease Status',weight='bold', size=18)
plt.ylabel('Count',weight='bold', size=18)
plt.yticks([0,max(status_counts.values)//2,max(status_counts.values)],size=14)
plt.xticks(size=14)
sns.despine()
plt.savefig("macro_disease_status_counts.pdf", bbox_inches="tight")
plt.show()